1. CASE FOLDING

In [ ]:
# Case Folding
import pandas as pd
import re
from google.colab import drive

def clean_text(text):
    # Menghapus karakter khusus, angka, dan simbol
    text = re.sub('[^a-zA-Z\s]', '', text)

    # Menghapus angka dari teks
    text = re.sub('\d', '', text)

    # Menghapus emoji (emoticon) tanpa mengganti dengan spasi
    text = re.sub(r'[^\w\s]', '', text)

    # Menghapus tiga atau lebih kemunculan karakter yang sama berurutan
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    return text

drive.mount('/content/gdrive')
file_path = '/content/gdrive/MyDrive/modelling/dataset_sementara.csv'

# Membaca file CSV ke dalam DataFrame
df = pd.read_csv(file_path, delimiter=';')

# Tahap 1: Case folding - Mengubah semua teks menjadi huruf kecil
df['comment_before'] = df['comment']
df['comment'] = df['comment'].str.lower()

# Menghapus spasi di awal dan akhir teks
df['comment'] = df['comment'].str.strip()

# Membersihkan teks menggunakan fungsi clean_text
df['comment'] = df['comment'].apply(clean_text)

# Menyimpan jumlah baris sebelum penghapusan baris kosong
jumlah_sebelum = len(df['comment_before'])

# Menghapus baris yang sepenuhnya kosong
df = df[df['comment'].str.strip() != '']

# Menyimpan jumlah baris setelah penghapusan baris kosong
jumlah_sesudah = len(df['comment'])

# Mengganti tanda (!) dan (.) dengan spasi
df['comment'] = df['comment'].replace(['!', '.'], ' ')

# Menghapus kata duplikat
df = df.drop_duplicates(subset=['comment'])

# Menampilkan hasil
print("Sebelum Case Folding:")
print(df['comment_before'].head())

print("\nSesudah Case Folding:")
print(df['comment'].head())

# Menampilkan perbandingan jumlah baris sebelum dan sesudah penghapusan baris kosong
print("\nJumlah Baris Sebelum Penghapusan Baris Kosong:", jumlah_sebelum)
print("Jumlah Baris Setelah Penghapusan Baris Kosong:", jumlah_sesudah)

# Menampilkan perbandingan jumlah baris sebelum dan sesudah penghapusan baris kosong
print("\nJumlah Baris Sebelum Penghapusan Baris Kosong:", jumlah_sebelum)
print("Jumlah Baris Setelah Penghapusan Baris Kosong:", jumlah_sesudah)

# Menampilkan jumlah baris setelah menghapus nilai NaN dan baris yang sesuai di kolom 'label'
jumlah_setelah_hapus_nan = len(df)
print("Jumlah Baris Setelah Menghapus Nilai NaN dan Baris yang Sesuai di Kolom 'label':", jumlah_setelah_hapus_nan)

# Menghapus nilai NaN dan baris yang sesuai di kolom 'label'
df = df.dropna(subset=['comment', 'label'])

# Menampilkan hasil
print("\nSetelah Menghapus Nilai NaN dan Baris yang Sesuai di Kolom 'label':")
print(df.head())


Mounted at /content/gdrive
Sebelum Case Folding:
0     Masih ada konten yang ENGGAK layak untuk anak ??
1    Mendidik anak saya... terimakasih YouTube kidd...
2    chanel yang dibuat untuk dewasa lalu dirubah m...
3    meski konten game gak semua anak anak, kalanga...
4    terus kalo kita seting biar anak anak sama ora...
Name: comment_before, dtype: object

Sesudah Case Folding:
0       masih ada konten yang enggak layak untuk anak 
1          mendidik anak saya terimakasih youtube kids
2    chanel yang dibuat untuk dewasa lalu dirubah m...
3    meski konten game gak semua anak anak kalangan...
4    terus kalo kita seting biar anak anak sama ora...
Name: comment, dtype: object

Jumlah Baris Sebelum Penghapusan Baris Kosong: 2184
Jumlah Baris Setelah Penghapusan Baris Kosong: 2184

Jumlah Baris Sebelum Penghapusan Baris Kosong: 2184
Jumlah Baris Setelah Penghapusan Baris Kosong: 2184
Jumlah Baris Setelah Menghapus Nilai NaN dan Baris yang Sesuai di Kolom 'label': 1530

Setelah Menghapu

2.TOKENIZING

In [ ]:
# Tokenizing
import nltk
from nltk.tokenize import word_tokenize

# Install library NLTK
!pip install nltk

# Download data pendukung untuk tokenizing
nltk.download('punkt')

# Mengaplikasikan tokenizing pada kolom 'comment'
df['tokens'] = df['comment'].apply(word_tokenize)

# Menampilkan DataFrame setelah tokenizing
print("DataFrame setelah tokenizing:")
print(df.head())


In [ ]:
# Menampilkan jumlah baris dan kolom DataFrame sebelum tokenizing
print("\nJumlah Baris dan Kolom Sebelum Tokenizing:", df.shape)

# Menghapus nilai NaN dan baris yang sesuai di kolom 'label' setelah tokenizing
df = df.dropna(subset=['tokens', 'label'])

# Menampilkan jumlah baris dan kolom DataFrame setelah menghapus nilai NaN dan baris yang sesuai di kolom 'label' setelah tokenizing
print("\nJumlah Baris dan Kolom Setelah Menghapus Nilai NaN dan Baris yang Sesuai di Kolom 'label' setelah Tokenizing:", df.shape)
print("\nDataFrame setelah menghapus nilai NaN dan baris yang sesuai di kolom 'label' setelah tokenizing:")
print(df.head())


3.SLANGWORDS

In [ ]:
# Slangword
import json

# Memuat kamus slangwords dari file
slangwords_file_path = '/content/gdrive/MyDrive/modelling/slang.json'
with open(slangwords_file_path, 'r', encoding='utf-8') as file:
    slangwords_list = json.load(file)

# Mengonversi list ke dictionary
slangwords_dict = {item["slang"]: item["formal"] for item in slangwords_list}

# Fungsi untuk menggantikan slangwords dalam kalimat
def replace_slangwords(sentence, slangwords_dict):
    return ' '.join(slangwords_dict.get(word, word) for word in sentence.split())

# Mengonversi list token ke dalam kalimat string sebelum penggantian slangwords
df['comment'] = df['tokens'].apply(lambda x: ' '.join(x))

# Menggantikan slangwords dalam kolom 'comment'
df['comment'] = df['comment'].apply(lambda x: replace_slangwords(x, slangwords_dict))

# Menghapus kolom 'tokens'
del df['tokens']

# Menampilkan DataFrame setelah menggantikan slangwords
print("DataFrame setelah menggantikan slangwords dan menghapus tokens:")
print(df.head())


DataFrame setelah menggantikan slangwords dan menghapus tokens:
                                             comment  label  \
0       masih ada konten yang tidak layak untuk anak      0   
1         mendidik anak aku terimakasih youtube kids      1   
2  chanel yang dibuat untuk dewasa lalu dirubah m...      0   
3  meski konten game tidak semua anak anak kalang...      0   
4  terus kalau kita seting biar anak anak sama or...      0   

                                      comment_before  
0   Masih ada konten yang ENGGAK layak untuk anak ??  
1  Mendidik anak saya... terimakasih YouTube kidd...  
2  chanel yang dibuat untuk dewasa lalu dirubah m...  
3  meski konten game gak semua anak anak, kalanga...  
4  terus kalo kita seting biar anak anak sama ora...  


4.STOPWORD REMOVAL

In [ ]:
# Stopword

!pip install Sastrawi

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Fungsi untuk melakukan stopword removal tanpa stemming menggunakan PySastrawi
def sastrawi_stopword_removal(text):
    # Menghapus stopword
    stopword_factory = StopWordRemoverFactory()
    stopword_remover = stopword_factory.create_stop_word_remover()
    text_without_stopword = stopword_remover.remove(text)

    return text_without_stopword

# Contoh penggunaan pada kolom 'comment' setelah menggantikan slangwords dan menghapus tokens
df['comment'] = df['comment'].apply(sastrawi_stopword_removal)

# Menampilkan DataFrame setelah menggantikan slangwords, menghapus tokens, dan melakukan stopword removal dengan PySastrawi
print("DataFrame setelah menggantikan slangwords, menghapus tokens, dan melakukan stopword removal:")
print(df.head())


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.5 MB/s eta 0:00:00
DataFrame setelah menggantikan slangwords, menghapus tokens, dan melakukan stopword removal:
                                             comment  label  \
0                        ada konten tidak layak anak      0   
1         mendidik anak aku terimakasih youtube kids      1   
2  chanel dibuat dewasa lalu dirubah menjadi kont...      0   
3  meski konten game semua anak anak kalangan dew...      0   
4  terus kalau seting biar anak anak sama orang d...      0   

                                      comment_before  
0   Masih ada konten yang ENGGAK layak untuk anak ??  
1  Mendidik anak saya... terimakasih YouTube kidd...  
2  chanel yang dibuat untuk dewasa lalu dirubah m...  
3  meski konten game gak semua anak anak, kalanga...  
4  terus kalo kita seting biar anak anak sama ora...  


5. STEMMMING

In [ ]:
# Stemming
!pip install Sastrawi

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Fungsi untuk melakukan stemming menggunakan PySastrawi
def sastrawi_stemming(text):
    # Membuat objek stemmer
    stemmer_factory = StemmerFactory()
    stemmer = stemmer_factory.create_stemmer()

    # Melakukan stemming
    stemmed_text = stemmer.stem(text)

    return stemmed_text

# Pemanggilan variabel comment untuk di lakukan stemming dengan PySastrawi
df['comment'] = df['comment'].apply(sastrawi_stemming)

# Menampilkan DataFrame setelah menggantikan slangwords, menghapus tokens, dan melakukan stemming dengan PySastrawi
print("DataFrame setelah menggantikan slangwords, menghapus tokens, dan melakukan stemming:")
print(df.head())


DataFrame setelah menggantikan slangwords, menghapus tokens, dan melakukan stemming:
                                             comment  label  \
0                        ada konten tidak layak anak      0   
1            didik anak aku terimakasih youtube kids      1   
2  chanel buat dewasa lalu rubah jadi konten anak...      0   
3  meski konten game semua anak anak kalang dewas...      0   
4  terus kalau ting biar anak anak sama orang dew...      0   

                                      comment_before  
0   Masih ada konten yang ENGGAK layak untuk anak ??  
1  Mendidik anak saya... terimakasih YouTube kidd...  
2  chanel yang dibuat untuk dewasa lalu dirubah m...  
3  meski konten game gak semua anak anak, kalanga...  
4  terus kalo kita seting biar anak anak sama ora...  


In [ ]:
# Menghapus kolom comment_before
df = df.drop(columns=['comment_before'])

# Menampilkan DataFrame setelah menghapus kolom comment_before
print("DataFrame setelah menghapus kolom comment_before:")
print(df.head())


DataFrame setelah menghapus kolom comment_before:
                                             comment  label
0                        ada konten tidak layak anak      0
1            didik anak aku terimakasih youtube kids      1
2  chanel buat dewasa lalu rubah jadi konten anak...      0
3  meski konten game semua anak anak kalang dewas...      0
4  terus kalau ting biar anak anak sama orang dew...      0


In [ ]:
# Menampilkan Jumlah Baris dan Kolom Sebelum Tokenizing
print(f"Jumlah Baris dan Kolom Sebelum Tokenizing: {df.shape}")

# Menampilkan Jumlah Baris dan Kolom Setelah Menghapus Nilai NaN dan Baris yang Sesuai di Kolom 'label' setelah Tokenizing
df = df.dropna(subset=['comment'])
print(f"Jumlah Baris dan Kolom Setelah Menghapus Nilai NaN dan Baris yang Sesuai di Kolom 'label' setelah Tokenizing: {df.shape}")

# Menampilkan Jumlah Baris Setelah Menghapus Baris Duplikat
df = df.drop_duplicates(subset=['comment'])
print(f"Jumlah Baris Setelah Menghapus Baris Duplikat: {df.shape}")

# Menampilkan Jumlah Baris Sebelum dan Setelah Menghapus Nilai NaN dan Baris yang Sesuai di Kolom 'label'
jumlah_setelah_hapus_nan = len(df)
print(f"Jumlah Baris Sebelum dan Setelah Menghapus Nilai NaN dan Baris yang Sesuai di Kolom 'label': ({jumlah_sebelum}, {jumlah_setelah_hapus_nan})")

# Menampilkan Jumlah Baris Setelah Menghapus Baris Kosong
print(f"Jumlah Baris Setelah Menghapus Baris Kosong: {jumlah_sesudah}")


Jumlah Baris dan Kolom Sebelum Tokenizing: (1530, 2)
Jumlah Baris dan Kolom Setelah Menghapus Nilai NaN dan Baris yang Sesuai di Kolom 'label' setelah Tokenizing: (1530, 2)
Jumlah Baris Setelah Menghapus Baris Duplikat: (1473, 2)
Jumlah Baris Sebelum dan Setelah Menghapus Nilai NaN dan Baris yang Sesuai di Kolom 'label': (2184, 1473)
Jumlah Baris Setelah Menghapus Baris Kosong: 2184


In [ ]:
# menyimpan file CSV di Google Drive

output_file_path = '/content/gdrive/MyDrive/modelling/preprocessed_data.csv'

# Menyimpan DataFrame ke file CSV
df.to_csv(output_file_path, index=False)

# Menampilkan pesan bahwa data berhasil disimpan
print(f"Data berhasil disimpan ke: {output_file_path}")

Data berhasil disimpan ke: /content/gdrive/MyDrive/modelling/preprocessed_data.csv


In [ ]:
# Melihat Tipe data dari kolom comment
tipe_data_comment = df['comment'].dtype

if tipe_data_comment == 'O':
    print("Tipe data dari kolom 'comment' adalah Object (objek)")
elif tipe_data_comment == 'string':
    print("Tipe data dari kolom 'comment' adalah String (string)")
else:
    print(f"Tipe data dari kolom 'comment': {tipe_data_comment}")


Tipe data dari kolom 'comment' adalah Object (objek)
